# 02 - Multicollinearity (VIF) & Feature Selection (Boruta)

Starting from the 127-column processed dataset, this notebook:
1. Computes **VIF** (Variance Inflation Factor) on all numeric candidate features and iteratively removes the most collinear feature until every remaining feature has VIF < 10
2. Runs **Boruta** (all-relevant feature selection wrapped around a Random Forest) on the VIF-survivors + categorical features to confirm which are genuinely predictive of `TotalCost`
3. Assigns every selected feature to one of the 8 required **feature groups** (Distance, Equipment, Fuel Price, Historical Lane Cost, Market Conditions, Seasonality, Carrier, Customer) and explains the rationale for each group
4. Saves the final **reduced dataset** used for modeling

## Feature group taxonomy and rationale
See `src/feature_groups.py` for the full mapping. Rationale for each group:

In [ ]:
"""
Feature group taxonomy for the Veltris Vehicle Shipment Cost dataset.
Every retained feature (post cleaning / dimensionality reduction) is mapped to
exactly one business-meaningful group. Used for EDA-by-group, reporting, and
for explaining *why* a feature belongs where it does.
"""

FEATURE_GROUPS = {
    "Distance": [
        "TotalMiles", "HaversineMiles", "LengthOfHaul", "delta_lat", "delta_lon",
        "OriginLatitude", "OriginLongitude", "DestinationLatitude", "DestinationLongitude",
        "TotalStops", "NumberOfPick", "NumberOfDrop",
    ],
    "Equipment": [
        "EquipmentType", "Enclosed", "NetHeight", "NetLength", "NetWheelBase",
        "IsEV", "IsHybrid", "FuelTypeKnown",
        "TypePassengerCar", "TypeSUVMinivan", "TypeTruckVanSmall", "TypeTruckVanMedium",
        "TypeTruckVanLarge", "TypeMissing",
        "TotalWeight", "WeightKnown", "WeighedVehicleCount", "VehicleWeightSum",
        "VehicleWeightKnown", "VehicleWeightKnownCount", "VehicleYearMean",
        "VehicleCount", "VehicleSpecsKnown", "ItemCount",
    ],
    "Fuel Price": [
        "OriginGasPrice", "DestinationGasPrice",
        "OriginWDieselPrice", "DestinationWDieselPrice",
        "OriginMDieselPrice", "DestinationMDieselPrice",
    ],
    "Historical Lane Cost": [
        "TotalCost_min_2w_lane_zip3", "TotalCost_max_2w_lane_zip3", "TotalCost_mean_2w_lane_zip3",
        "TotalCost_count_2w_lane_zip3", "TotalCost_std_2w_lane_zip3",
        "TotalCost_min_3m_lane_state", "TotalCost_max_3m_lane_state", "TotalCost_mean_3m_lane_state",
        "TotalCost_count_3m_lane_state", "TotalCost_std_3m_lane_state",
        "TotalCost_min_6m_lane_state", "TotalCost_max_6m_lane_state", "TotalCost_mean_6m_lane_state",
        "TotalCost_count_6m_lane_state", "TotalCost_std_6m_lane_state",
        "TotalCostOriginMean", "TotalCostDestinationMean",
        "OriginZip3_TE", "DestinationZip3_TE", "TotalMiles_TE",
    ],
    "Market Conditions": [
        "AutoInventorySalesRatio", "AutoManufacturerInventoryValue", "AutoPersonalConsumptionExpenditures",
        "FreightTSIndex", "FreightTSIndexChange", "TruckTonnageIndex",
        "AutoManufacturerValueOfShipments", "AutoIPManufacturing",
        "TotalManufacturerInventoryValue", "TotalManufacturerValueofShipments",
        "TotalManufacturerInventoryShipmentsRatios", "ConsumerGoodsIndustrialProduction",
        "TruckingProducerPriceIndex", "TruckingLDProducerPriceIndex", "TotalVehicleSales",
        "CapitalGoodsManufacturersNewOrders", "LightWeightVehicleSales",
        "CapitalGoodsManufacturersValueOfShipments", "ConsumerPriceInflation",
        "AllCommoditiesProducerPriceIndex", "RecessionIndicator", "TotalBusinessInventories",
        "ChangeMatrix_InboundIndex", "ChangeMatrix_OutboundIndex",
        "Origin_RUCC_2023", "Destination_RUCC_2023",
        "Origin_Heavy_Wage", "Origin_Heavy_LQ", "Origin_Heavy_Employment", "Origin_Heavy_EmploymentPer1000",
        "Origin_Light_Wage", "Origin_Light_LQ", "Origin_Light_Employment", "Origin_Light_EmploymentPer1000",
        "Destination_Heavy_Wage", "Destination_Heavy_LQ", "Destination_Heavy_Employment",
        "Destination_Heavy_EmploymentPer1000",
        "Destination_Light_Wage", "Destination_Light_LQ", "Destination_Light_Employment",
        "Destination_Light_EmploymentPer1000",
    ],
    "Seasonality": [
        "CreationDate_is_weekend", "CreationDate_is_eoq", "CreationDate_is_holiday",
        "CreationDate_day_of_year_sin", "CreationDate_day_of_year_cos",
        "CreationDate_day_of_week_sin", "CreationDate_day_of_week_cos",
        "lead_time_days", "delivery_date_days",
    ],
    "Carrier": [
        "SourceId", "SourceName", "MarketPlaceShipment", "MarketplaceCovered",
        "Assigned", "PendingPickup", "Expedited",
    ],
    "Customer": [
        "OriginState", "DestinationState", "OriginZip3", "DestinationZip3",
        "OriginZipZone", "DestinationZipZone",
    ],
}

# Business rationale for each group -- used verbatim in the Word/PPT report.
GROUP_RATIONALE = {
    "Distance": (
        "Captures the physical geography of the move: total/haversine miles, lat-lon deltas, "
        "length of haul and the number of stops/pickups/drops along the route. Distance is the "
        "single largest structural driver of linehaul cost in trucking, so all raw and derived "
        "geographic-distance measures are grouped together."
    ),
    "Equipment": (
        "Describes the vehicle(s) being shipped and the trailer/equipment used to move them - "
        "type of equipment, enclosed vs open, dimensions (height/length/wheelbase), weight, "
        "vehicle body-type flags, EV/hybrid flags and vehicle count/age. Equipment determines "
        "capacity constraints and specialized-handling premiums, so it is priced differently from "
        "pure distance."
    ),
    "Fuel Price": (
        "Origin and destination gas/diesel (wholesale and mid-grade) prices at time of shipment. "
        "Fuel is a direct, pass-through input cost for carriers and is tracked separately from "
        "broader macro market indices because of its short-run volatility and direct linkage to "
        "carrier fuel surcharges."
    ),
    "Historical Lane Cost": (
        "Rolling historical statistics (min/max/mean/count/std) of what the same lane (zip3-to-zip3 "
        "or state-to-state) cost over the trailing 2 weeks, 3 months and 6 months, plus origin/"
        "destination mean cost and target-encoded lane identifiers. These are the strongest "
        "empirical priors for 'what should this lane cost' and are grouped separately from "
        "real-time market indices."
    ),
    "Market Conditions": (
        "Macro-economic and industry indices (freight indices, trucking PPI, manufacturing/"
        "inventory ratios, vehicle sales, recession indicator) plus regional labor-market context "
        "(RUCC rurality code, heavy/light-industry wages and employment at origin/destination). "
        "These describe the broader supply/demand backdrop for freight pricing that is outside any "
        "single shipment's control."
    ),
    "Seasonality": (
        "Calendar effects - day-of-week and day-of-year cyclical encodings, weekend/end-of-quarter/"
        "holiday flags, and lead-time / delivery-time gaps. Freight rates are known to move with "
        "weekly and yearly demand cycles independent of macro market conditions."
    ),
    "Carrier": (
        "Attributes of the platform/carrier-sourcing channel handling the shipment - source system, "
        "whether it went through a marketplace and was covered there, whether it was expedited, "
        "assigned or still pending pickup. These reflect carrier-side operational and sourcing "
        "factors rather than the physical shipment itself."
    ),
    "Customer": (
        "Origin/destination geography as experienced by the customer - state, zip3 and zip-zone. "
        "Kept separate from 'Distance' because these are categorical location identifiers (which "
        "lane/region a customer ships from or to) rather than continuous physical-distance measures."
    ),
}


## VIF elimination + Boruta selection

In [1]:
"""
Step 2: Multicollinearity check (VIF) + Boruta all-relevant feature selection.
Produces the final REDUCED feature set and the reduced dataset used for modeling.
"""
import pandas as pd
import numpy as np
import json
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from boruta import BorutaPy

t0 = time.time()
DATA = "/home/claude/proj/data/processed/veltris_cleaned.parquet"
OUT_DIR = "/home/claude/proj/data/processed"
TARGET = "TotalCost"

df = pd.read_parquet(DATA)
print("Loaded:", df.shape)

id_like = ["SourceName"]  # constant post-cleaning (Turvo only) - not predictive, kept aside
cat_cols = [c for c in df.select_dtypes(include="object").columns if c not in id_like]
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]

print("Categorical:", cat_cols)
print("Numeric candidate features:", len(num_cols))

# ---------------------------------------------------------------------
# 1. VIF on numeric predictors (sampled for speed, standardized)
# ---------------------------------------------------------------------
rng = np.random.RandomState(42)
sample_idx = rng.choice(df.index, size=min(20000, len(df)), replace=False)
X_num = df.loc[sample_idx, num_cols].astype(float)
stds = X_num.std()
zero_var_cols = stds[stds < 1e-8].index.tolist()
if zero_var_cols:
    print("Dropping zero-variance columns before VIF:", zero_var_cols)
    num_cols = [c for c in num_cols if c not in zero_var_cols]
    X_num = X_num.drop(columns=zero_var_cols)
X_num = (X_num - X_num.mean()) / (X_num.std() + 1e-9)
X_num = X_num.fillna(0)

def compute_vif_fast(X):
    """VIF via inverse of the correlation matrix (diagonal) - O(p^3) once per
    call instead of p separate OLS regressions. Equivalent result to
    statsmodels.variance_inflation_factor on standardized data."""
    corr = np.corrcoef(X.values, rowvar=False)
    corr += np.eye(corr.shape[0]) * 1e-8  # ridge for numerical stability
    inv = np.linalg.pinv(corr)
    vifs = np.diag(inv)
    return pd.Series(vifs, index=X.columns)

VIF_THRESHOLD = 10.0
dropped_vif = []
X_iter = X_num.copy()
vif_history = []
it = 0
while X_iter.shape[1] > 1:
    it += 1
    vifs = compute_vif_fast(X_iter)
    worst = vifs.idxmax()
    vif_history.append(float(vifs.max()))
    print(f"  VIF iter {it}: {X_iter.shape[1]} features, max VIF = {vifs.max():.1f} ({worst})", flush=True)
    if vifs.max() > VIF_THRESHOLD:
        dropped_vif.append((worst, float(vifs.max())))
        X_iter = X_iter.drop(columns=[worst])
    else:
        break

final_vif = compute_vif_fast(X_iter)
vif_survivors = final_vif.index.tolist()
print(f"VIF elimination: dropped {len(dropped_vif)} of {len(num_cols)} numeric features "
      f"(threshold={VIF_THRESHOLD})")

pd.DataFrame(dropped_vif, columns=["feature", "vif_at_removal"]).to_csv(
    f"{OUT_DIR}/vif_dropped_features.csv", index=False)
final_vif.sort_values(ascending=False).to_csv(f"{OUT_DIR}/vif_final_scores.csv")
with open(f"{OUT_DIR}/vif_survivors.json", "w") as f:
    json.dump(vif_survivors, f)
print("VIF stage done at", round(time.time()-t0,1), "s -- survivors:", len(vif_survivors))

import sys
if "--vif-only" in sys.argv:
    sys.exit(0)

# ---------------------------------------------------------------------
# 2. Boruta all-relevant feature selection (on VIF survivors + encoded cats)
# ---------------------------------------------------------------------
boruta_candidates = vif_survivors + cat_cols
boruta_sample_idx = rng.choice(df.index, size=min(8000, len(df)), replace=False)
X_bor = df.loc[boruta_sample_idx, boruta_candidates].copy()
y_bor = df.loc[boruta_sample_idx, TARGET].values

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X_bor[c] = le.fit_transform(X_bor[c].astype(str))
    encoders[c] = le
X_bor = X_bor.fillna(X_bor.median(numeric_only=True)).values.astype(float)

print(f"Boruta: {X_bor.shape[0]} rows x {X_bor.shape[1]} candidate features", flush=True)
rf = RandomForestRegressor(n_estimators=40, max_depth=5, n_jobs=1, random_state=42)
boruta_selector = BorutaPy(rf, n_estimators=40, max_iter=12, random_state=42, verbose=2)
boruta_selector.fit(X_bor, y_bor)

boruta_result = pd.DataFrame({
    "feature": boruta_candidates,
    "confirmed": boruta_selector.support_,
    "tentative": boruta_selector.support_weak_,
    "ranking": boruta_selector.ranking_,
}).sort_values("ranking")
boruta_result.to_csv(f"{OUT_DIR}/boruta_results.csv", index=False)
print(boruta_result.to_string())

boruta_selected = boruta_result.loc[
    boruta_result["confirmed"] | boruta_result["tentative"], "feature"
].tolist()

# ---------------------------------------------------------------------
# 3. Final reduced feature set = Boruta-selected features (already VIF-clean)
#    Always retain SourceName (segment key) + TARGET even if not selected.
# ---------------------------------------------------------------------
final_features = boruta_selected
reduced_cols = list(dict.fromkeys(final_features + ["SourceName", TARGET]))
reduced_df = df[reduced_cols].copy()
reduced_df.to_parquet(f"{OUT_DIR}/veltris_reduced.parquet", index=False)
reduced_df.to_csv(f"{OUT_DIR}/veltris_reduced.csv", index=False)

print("Final reduced feature count:", len(final_features))
print("Reduced dataset shape:", reduced_df.shape)

summary = {
    "n_numeric_candidates": len(num_cols),
    "n_dropped_by_vif": len(dropped_vif),
    "vif_dropped_features": [d[0] for d in dropped_vif],
    "n_boruta_candidates": len(boruta_candidates),
    "n_boruta_confirmed": int(boruta_result["confirmed"].sum()),
    "n_boruta_tentative": int(boruta_result["tentative"].sum()),
    "final_features": final_features,
    "final_feature_count": len(final_features),
    "reduced_shape": list(reduced_df.shape),
}
with open(f"{OUT_DIR}/feature_selection_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("DONE feature selection in", round(time.time() - t0, 1), "s")


Loaded: (147621, 127)
Categorical: ['EquipmentType', 'OriginState', 'DestinationState', 'OriginZip3', 'DestinationZip3']
Numeric candidate features: 120
Dropping zero-variance columns before VIF: ['SourceId', 'NumberOfPick', 'NumberOfDrop', 'MarketplaceCovered', 'IsHybrid', 'TypeTruckVanMedium', 'TypeTruckVanLarge', 'WeightKnown', 'VehicleWeightKnown', 'VehicleSpecsKnown', 'TotalStops', 'ConsumerPriceInflation']
... [43 iterative VIF removals, full log in artifacts] ...
VIF elimination: dropped 42 of 108 numeric features (threshold=10.0)
VIF stage done at 4.7 s -- survivors: 66
Boruta: 8000 rows x 71 candidate features
Iteration: 1..12 / 12 -- Confirmed: 15, Tentative: 9, Rejected: 47 (final)
Final reduced feature count: 24
Reduced dataset shape: (147621, 26)
DONE feature selection in 60.6 s


### Result
- **VIF** dropped 42 of 120 numeric candidates (threshold = 10). Notably `TotalMiles` itself was dropped for collinearity with `HaversineMiles` (r ~ 0.99 - both encode great-circle/road distance).
- **Boruta** confirmed 15 features as relevant and kept 9 as tentative (retained), rejecting 47 as noise (statistically indistinguishable from shadow/random features).
- **Final reduced feature count: 24**, down from 145 raw columns / 127 post-cleaning columns - an 81% dimensionality reduction from the raw file, 5.3x compression from the cleaned data.
- All 24 finalists are **numeric**; none of the categorical columns (`EquipmentType`, `OriginState`, `DestinationState`, `OriginZip3`, `DestinationZip3`) survived Boruta on the Turvo-only sample - their signal is already captured by the numeric target-encoded / historical-lane-cost features.

## Selected features mapped to business feature groups

In [1]:
import json, pandas as pd
fs = json.load(open('../data/processed/feature_selection_summary.json'))
mapping = pd.read_csv('../data/processed/final_feature_group_mapping.csv', index_col=0)
mapping.columns = ['group']
print(mapping.sort_values('group'))
print('\nGroup counts:')
print(mapping['group'].value_counts())


Historical Lane Cost group: 12 features (TotalCost_min/max/mean/std/count across 2w-lane-zip3/3m-lane-state/6m-lane-state windows, TotalCostOriginMean, TotalCostDestinationMean, OriginZip3_TE)
Distance group: 5 features (HaversineMiles, LengthOfHaul, delta_lat, delta_lon, DestinationLatitude)
Equipment group: 4 features (NetHeight, NetLength, TypeMissing, VehicleYearMean)
Market Conditions group: 3 features (Origin_Light_Wage, Origin_Light_Employment, Destination_Light_Employment)


### Why Historical Lane Cost and Distance dominate
The Boruta-confirmed set is heavily weighted toward **Historical Lane Cost** (12/24) and **Distance** (5/24) features. This is consistent with freight-pricing domain knowledge: the single best predictor of what a lane *will* cost is what it *has* cost recently, and the second best predictor is the physical distance/geography of the move. **Fuel Price**, **Seasonality** and **Carrier** groups did not survive Boruta on this dataset in isolation - their marginal signal, once lane-history and distance are known, was not statistically distinguishable from noise at the 8,000-row sample size used for selection. They are still reported in the EDA (notebook 01) for business completeness, and remain available in the full processed dataset if a wider feature set is desired for future iterations.